In [21]:
# Step 1: LLM
from langchain.chat_models import init_chat_model

llm_model = init_chat_model(model="gpt-4o")

In [22]:
from langchain_core.messages import trim_messages

trimmer = trim_messages(
    max_tokens=200,          # token budget for the trimmed history
    strategy="last",          # keep the most recent messages
    token_counter=llm_model,  # uses the model's tokenizer to count accurately
    include_system=True,      # keep the system message even if trimming
    start_on="human",         # trimmed history should start on a human message (avoids orphaned AI replies)
)

In [23]:
# Step 2: Prompt — no {context}, just history + question
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful, friendly assistant. Answer naturally, using "
               "the conversation history to maintain context."),
    MessagesPlaceholder("chat_history"),
    ("human", "{question}"),
])

In [24]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

In [25]:
# Step 3: Simple chain
from langchain_core.runnables import RunnablePassthrough

base_chain = (
    RunnablePassthrough.assign(chat_history=lambda x: trimmer.invoke(x["chat_history"]))
    | prompt_template
    | llm_model
    | parser
)

In [26]:
# Step 4: In-memory store for chat histories, keyed by session_id
from langchain_core.chat_history import InMemoryChatMessageHistory

store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

In [27]:
# Step 5: Wrap with memory
from langchain_core.runnables.history import RunnableWithMessageHistory

chain_with_memory = RunnableWithMessageHistory(
    base_chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history",
)

c:\Users\Dell\Desktop\RAG Course\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [28]:
config = {"configurable": {"session_id": "trim_test"}}

In [29]:
# Simulate a long conversation
topics = [
    "My name is Acchint.",
    "I'm learning about RAG pipelines.",
    "I built a Chroma vectorstore earlier.",
    "I also implemented MultiQueryRetriever.",
    "Then I added contextual compression.",
    "Now I'm working on conversation memory.",
]

for msg in topics:
    chain_with_memory.invoke({"question": msg}, config=config)

In [30]:

# Full stored history
history = get_session_history("trim_test")
print(f"Full history: {len(history.messages)} messages")
for msg in history:
    print(f"{msg}")

Full history: 12 messages
('messages', [HumanMessage(content='My name is Acchint.', additional_kwargs={}, response_metadata={}), AIMessage(content='Nice to meet you, Acchint! How can I assist you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content="I'm learning about RAG pipelines.", additional_kwargs={}, response_metadata={}), AIMessage(content="That's great! RAG (Retrieval-Augmented Generation) pipelines are a powerful approach in natural language processing that combine retrieval-based methods with generative models. Essentially, they first retrieve relevant documents from a large dataset and then generate a response based on both the retrieved documents and the input query. This can help improve the accuracy and relevance of the generated responses.\n\nAre you working on a specific project or just exploring the concept? Let me know if you have any specific questions!", additional_kwargs={}, response_metadata={}, tool_call

In [31]:

# What the trimmer would actually send to the LLM
trimmed = trimmer.invoke(history.messages)
print(f"Trimmed history: {len(trimmed)} messages")
for msg in trimmed:
    print(f"{msg.type}: {msg.content}")

Trimmed history: 4 messages
human: Then I added contextual compression.
ai: Adding contextual compression is a smart move! Contextual compression can help reduce the amount of irrelevant or redundant information retrieved, focusing instead on the most important parts of each document based on the context of the query. This can make the retrieval results more concise and relevant, ultimately improving user satisfaction and the efficiency of processing the results.

How has contextual compression impacted the performance or quality of your retrieval results? Have you observed any specific benefits or challenges?
human: Now I'm working on conversation memory.
ai: Integrating conversation memory is a great step for creating a more interactive and human-like experience. By maintaining context over the course of a conversation, you can provide more relevant and coherent responses, anticipate user needs, and personalize interactions.

What approach are you taking to implement conversation mem

### Windowed Memory

In [32]:
class WindowedChatMessageHistory(InMemoryChatMessageHistory):
    def __init__(self, k: int = 6):
        super().__init__()
        self.k = k  # number of messages to keep (not exchanges — raw message count)

    def add_message(self, message):
        super().add_message(message)
        # Keep only the last k messages
        self.messages = self.messages[-self.k:]

In [33]:
config = {"configurable": {"session_id": "window_test"}}

In [35]:
from langchain_core.messages import trim_messages

windowed_trimmer = trim_messages(
    max_tokens=6,        # here this means "max 6 messages", not tokens
    strategy="last",
    token_counter=len,   # len() counts messages, not tokens
    include_system=True,
    start_on="human",
)

In [ ]:

base_chain = (
    RunnablePassthrough.assign(chat_history=lambda x: windowed_trimmer.invoke(x["chat_history"]))
    | prompt_template
    | llm_model
    | parser
)

chain_with_memory = RunnableWithMessageHistory(
    base_chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history",
)

chain_with_memory.invoke({"question": "My name is Acchint."}, config=config)
chain_with_memory.invoke({"question": "I like RAG pipelines."}, config=config)
chain_with_memory.invoke({"question": "I built a Chroma vectorstore."}, config=config)
chain_with_memory.invoke({"question": "I implemented MultiQueryRetriever."}, config=config)

# confirm the model "forgets" the name after enough turns
response = chain_with_memory.invoke({"question": "What's my name?"}, config=config)
print(response)

c:\Users\Dell\Desktop\RAG Course\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


I'm sorry, but I don't have access to personal data about users unless it has been shared with me in the course of our conversation. If you'd like, you can let me know your name!


### Summary Memory

In [37]:
from langchain_core.prompts import ChatPromptTemplate

summary_prompt = ChatPromptTemplate.from_template(
    """Summarize the following conversation concisely, preserving key facts,
names, decisions, and topics that might be needed later:

{conversation}

Concise summary:"""
)

In [38]:
from langchain_core.messages import SystemMessage, get_buffer_string

def summarize_messages(messages, llm, existing_summary: str = "") -> str:
    conversation_text = get_buffer_string(messages)
    if existing_summary:
        conversation_text = f"Previous summary: {existing_summary}\n\nNew messages:\n{conversation_text}"
    return llm.invoke(summary_prompt.format(conversation=conversation_text)).content

In [39]:
session_summaries = {}  # session_id -> running summary string
MAX_MESSAGES_BEFORE_SUMMARY = 6  # summarize once history exceeds this many messages
KEEP_RECENT = 2  # always keep the last N messages verbatim

def get_summarized_history(x):
    session_id = x["session_id"]
    messages = x["chat_history"]

    if len(messages) <= MAX_MESSAGES_BEFORE_SUMMARY:
        return messages  # short enough, no summarization needed yet

    to_summarize = messages[:-KEEP_RECENT]
    recent = messages[-KEEP_RECENT:]

    existing_summary = session_summaries.get(session_id, "")
    new_summary = summarize_messages(to_summarize, llm_model, existing_summary)
    session_summaries[session_id] = new_summary

    return [SystemMessage(content=f"Summary of earlier conversation: {new_summary}")] + recent

In [40]:
base_chain = (
    RunnablePassthrough.assign(chat_history=get_summarized_history)
    | prompt_template
    | llm_model
    | parser
)

chain_with_memory = RunnableWithMessageHistory(
    base_chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history",
)

c:\Users\Dell\Desktop\RAG Course\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [41]:
config = {"configurable": {"session_id": "summary_test"}}

In [42]:
topics = [
    "My name is Acchint.",
    "I'm learning about RAG pipelines.",
    "I built a Chroma vectorstore earlier.",
    "I also implemented MultiQueryRetriever.",
    "Then I added contextual compression.",
]

for msg in topics:
    chain_with_memory.invoke(
        {"question": msg, "session_id": "summary_test"},
        config=config
    )

# The critical test — does it still remember the name after summarization kicked in?
response = chain_with_memory.invoke(
    {"question": "What's my name?", "session_id": "summary_test"},
    config=config
)
print(response)

Your name is Acchint. If there's anything else you'd like to discuss or any questions you have, feel free to ask!


In [43]:
print("Stored summary:")
print(session_summaries.get("summary_test", "(none yet)"))

print("\nFull raw history still stored:")
history = get_session_history("summary_test")
print(f"{len(history.messages)} messages")

Stored summary:
Acchint is learning about Retrieval-Augmented Generation (RAG) pipelines and has built a Chroma vectorstore, which enhances retrieval efficiency in NLP tasks by storing text vectors for similarity search. Key considerations for Chroma include indexing, normalization, dimensionality reduction, and integration with generation models. Acchint also implemented a MultiQueryRetriever, which enhances retrieval by generating diverse queries, improving information retrieval, and ultimately providing a richer context for the generation component in the RAG system.

Full raw history still stored:
12 messages
